# PyTorch Tutorial - Part 2: Intermediate to Advanced

This is Part 2. It builds directly on Part 1.

Make sure you are comfortable with:
- Tensors, Autograd, and nn.Module
- The training loop (forward -> loss -> backward -> update)
- DataLoaders and saving models

Topics covered:
1. Convolutional Neural Networks (CNNs) for image recognition
2. RNNs and LSTMs for sequential data
3. Transfer Learning - reuse powerful pretrained models
4. Advanced Training - early stopping, mixed precision, gradient accumulation
5. Custom Image Datasets from folders
6. A reusable Trainer class for clean production-level training

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


Using device: cpu


## Section 1 - Convolutional Neural Networks (CNNs)

The problem with regular neural networks for images:
A 224x224 color image has 224 x 224 x 3 = 150,528 pixels.
If we use a Linear layer with 512 hidden neurons, the first layer alone has 77 million parameters.
That is too slow to train, prone to overfitting, and ignores spatial structure.

How CNNs solve this:
A convolutional layer uses small filters (like 3x3 patches) that slide across the image.
Each filter learns to detect a specific pattern (edge, curve, texture).
The same filter is reused everywhere - this is called weight sharing and dramatically reduces parameters.

CNN building blocks:
- Conv2d: applies filters to detect patterns - the main learning layer
- BatchNorm2d: normalizes feature maps for stable training
- ReLU: keeps only positive activations
- MaxPool2d: shrinks the spatial size, keeps strongest activations
- Flatten + Linear: converts 2D features to a class prediction

Output size formula: floor((input + 2*padding - kernel_size) / stride + 1)

In [2]:
# Understanding Conv2d
# nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
# in_channels:  number of input channels (1 = grayscale, 3 = RGB)
# out_channels: number of filters to learn
# kernel_size:  size of each filter (3 means 3x3)
# padding=1:    adds zeros around the border to keep output same size as input

conv = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)

# Test with a fake batch of images: [batch, channels, height, width]
fake_images = torch.randn(8, 3, 32, 32)  # 8 images, RGB, 32x32
output = conv(fake_images)

print('Input shape: ', fake_images.shape)   # [8, 3, 32, 32]
print('Output shape:', output.shape)         # [8, 32, 32, 32]

print(f'\nFilter weight shape: {conv.weight.shape}')
# [32, 3, 3, 3] = 32 filters, each looking at 3 channels, 3x3 spatial

total_params = sum(p.numel() for p in conv.parameters())
print(f'Parameters in this conv layer: {total_params}')
# Only 896 parameters - much fewer than a Linear layer!


Input shape:  torch.Size([8, 3, 32, 32])
Output shape: torch.Size([8, 32, 32, 32])

Filter weight shape: torch.Size([32, 3, 3, 3])
Parameters in this conv layer: 896


In [3]:
# A reusable ConvBlock
# In CNN architectures, the same pattern repeats:
# Conv2d -> BatchNorm -> ReLU -> (optional MaxPool)

class ConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels, kernel_size=3,
                 stride=1, padding=1, use_pool=False):
        super(ConvBlock, self).__init__()

        # bias=False because BatchNorm handles the shift
        self.conv = nn.Conv2d(in_channels, out_channels,
                              kernel_size, stride, padding, bias=False)
        self.bn   = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # MaxPool halves the spatial dimensions (optional)
        self.pool = nn.MaxPool2d(2, 2) if use_pool else None

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        if self.pool:
            x = self.pool(x)
        return x


# Test
block = ConvBlock(in_channels=3, out_channels=16, use_pool=True)
test_input = torch.randn(4, 3, 32, 32)
output = block(test_input)
print(f'ConvBlock input:  {test_input.shape}')  # [4, 3, 32, 32]
print(f'ConvBlock output: {output.shape}')       # [4, 16, 16, 16] - halved by pool


ConvBlock input:  torch.Size([4, 3, 32, 32])
ConvBlock output: torch.Size([4, 16, 16, 16])


In [4]:
# A complete CNN for image classification

class SimpleCNN(nn.Module):
    """
    CNN for image classification.
    Designed for small images (32x32) like CIFAR-10.

    Data flow:
    [B, 3, 32, 32] -> ConvBlock1 -> [B, 32, 16, 16]
                   -> ConvBlock2 -> [B, 64, 8, 8]
                   -> ConvBlock3 -> [B, 128, 4, 4]
                   -> GlobalAvgPool -> [B, 128]
                   -> FC layers -> [B, num_classes]
    """

    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        # Feature extraction: each block doubles channels, halves spatial size
        self.features = nn.Sequential(
            ConvBlock(3,   32,  use_pool=True),   # 32x32 -> 16x16
            ConvBlock(32,  64,  use_pool=True),   # 16x16 -> 8x8
            ConvBlock(64,  128, use_pool=True),   # 8x8   -> 4x4
        )

        # Global Average Pooling: averages each feature map to a single number
        # Better than flattening - reduces overfitting and works with any input size
        self.gap = nn.AdaptiveAvgPool2d((1, 1))  # output is always [B, C, 1, 1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


cnn = SimpleCNN(num_classes=10).to(device)
print(cnn)

params = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {params:,}')

dummy_imgs = torch.randn(16, 3, 32, 32).to(device)
logits = cnn(dummy_imgs)
print(f'Input: {dummy_imgs.shape} -> Output: {logits.shape}')  # [16, 10]


SimpleCNN(
  (features): Sequential(
    (0): ConvBlock(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): ConvBlock(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): ConvBlock(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, d

In [5]:
# Residual Connections - the key idea behind ResNet

# Problem with very deep networks:
# As gradients flow backward through many layers they shrink to near-zero.
# This is called the vanishing gradient problem - layers stop learning.

# Solution: skip connections (residual connections)
# Instead of: output = F(input)          (normal)
# We do:      output = F(input) + input  (residual)
# The shortcut provides a direct gradient path backward.
# This lets us train networks with 100+ layers.

class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x  # save the input as the shortcut

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Add the original input - this is the skip connection
        out = out + residual

        # Apply activation after the addition
        out = F.relu(out)

        return out


res_block = ResidualBlock(channels=64).to(device)
test_feat = torch.randn(4, 64, 16, 16).to(device)
out = res_block(test_feat)
print(f'ResidualBlock: input {test_feat.shape} -> output {out.shape}')
# Shape does not change - input and output have the same dimensions


ResidualBlock: input torch.Size([4, 64, 16, 16]) -> output torch.Size([4, 64, 16, 16])


## Section 2 - RNNs and LSTMs: Learning from Sequences

Sequential data has an order that matters:
- Text: 'The cat sat' - the word 'sat' depends on what came before
- Time series: today's stock price depends on the past few days
- Audio: each sample depends on previous samples

Regular neural networks treat each input independently - no memory of what came before.
This is a problem for sequential data.

A Recurrent Neural Network (RNN) has a hidden state - a kind of memory that gets updated at each step.
At each step: hidden_state = f(input, previous_hidden_state)

Problem with basic RNNs:
They struggle to remember information from far back in a sequence.

LSTM - Long Short-Term Memory:
LSTMs solve this with gates that learn what to forget, what to store, and what to output.
LSTMs can remember context across hundreds of steps.

In [6]:
# RNN input shape: [sequence_length, batch_size, input_size]
# sequence_length: how many time steps (words, frames, etc.)
# batch_size:      how many sequences to process in parallel
# input_size:      how many features per time step

seq_len    = 20
batch      = 32
input_size = 50

fake_sequence = torch.randn(seq_len, batch, input_size)
print(f'Input shape: {fake_sequence.shape}')  # [20, 32, 50]

# Basic RNN
rnn = nn.RNN(
    input_size=input_size,
    hidden_size=64,     # size of hidden state (the memory)
    num_layers=2,       # stack 2 RNN layers
    dropout=0.2,        # dropout between stacked layers
    batch_first=False   # input is [seq, batch, features]
)

output, hidden = rnn(fake_sequence)

print(f'\nRNN output shape: {output.shape}')
# [20, 32, 64] - hidden state at each time step, for each sample

print(f'RNN hidden shape: {hidden.shape}')
# [2, 32, 64] - final hidden state (one per layer)

print('\nWhen to use which output:')
print('  output:     for sequence-to-sequence tasks (e.g. translation)')
print('  hidden[-1]: for sequence-to-one tasks (e.g. classification)')


Input shape: torch.Size([20, 32, 50])

RNN output shape: torch.Size([20, 32, 64])
RNN hidden shape: torch.Size([2, 32, 64])

When to use which output:
  output:     for sequence-to-sequence tasks (e.g. translation)
  hidden[-1]: for sequence-to-one tasks (e.g. classification)


In [7]:
# LSTM: better memory for long sequences

lstm = nn.LSTM(
    input_size=50,
    hidden_size=64,
    num_layers=2,
    dropout=0.2,
    bidirectional=False,  # if True, processes sequence forward and backward
    batch_first=False
)

# LSTM returns two states:
# h_n: hidden state (short-term memory) - same as RNN
# c_n: cell state (long-term memory) - LSTM's special addition
output, (h_n, c_n) = lstm(fake_sequence)

print(f'LSTM output shape: {output.shape}')  # [20, 32, 64]
print(f'LSTM h_n shape:    {h_n.shape}')     # [2, 32, 64]
print(f'LSTM c_n shape:    {c_n.shape}')     # [2, 32, 64]

# Bidirectional LSTM: processes sequence both forwards and backwards
# Doubles the output size but gives the model context from both directions
bilstm = nn.LSTM(input_size=50, hidden_size=64, num_layers=1, bidirectional=True)
output_bi, (h_bi, c_bi) = bilstm(fake_sequence)

print(f'\nBidirectional LSTM output shape: {output_bi.shape}')
# [20, 32, 128] - 128 = 64 (forward) + 64 (backward)


LSTM output shape: torch.Size([20, 32, 64])
LSTM h_n shape:    torch.Size([2, 32, 64])
LSTM c_n shape:    torch.Size([2, 32, 64])

Bidirectional LSTM output shape: torch.Size([20, 32, 128])


In [8]:
# Complete LSTM Classifier for Sequence Data

class LSTMClassifier(nn.Module):
    """
    Classifies a sequence into one of several categories.
    Example uses: sentiment analysis, activity recognition, fault detection.
    """

    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(LSTMClassifier, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True  # easier: [batch, seq, features]
        )

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: [batch, seq_len, input_size]
        lstm_out, (h_n, c_n) = self.lstm(x)

        # For classification we only need the last time step's output
        last_output = lstm_out[:, -1, :]  # [batch, hidden_size]

        out = self.dropout(last_output)
        out = self.classifier(out)

        return out  # [batch, num_classes]


lstm_model = LSTMClassifier(
    input_size=20, hidden_size=64, num_layers=2, num_classes=5, dropout=0.3
).to(device)

# Test: 16 sequences, each 30 steps long, 20 features per step
test_seq = torch.randn(16, 30, 20).to(device)
output = lstm_model(test_seq)
print(f'Input shape: {test_seq.shape}')
print(f'Output shape: {output.shape}')  # [16, 5]


Input shape: torch.Size([16, 30, 20])
Output shape: torch.Size([16, 5])


## Section 3 - Transfer Learning

Training a large neural network from scratch requires millions of labeled images and days of GPU time.

Transfer Learning lets you skip that by starting with a pretrained model - a network already trained on
a massive dataset like ImageNet (1.2 million images, 1000 categories).

The pretrained network already knows how to detect edges, textures, shapes, and complex objects.
We reuse those learned features for our own task.

Three strategies:
1. Feature extraction: freeze all pretrained layers, train only a new output head. For small datasets.
2. Full fine-tuning: unfreeze everything and train all layers. Use a small learning rate.
3. Partial fine-tuning: freeze early layers, train later ones. For medium-sized datasets.

In [9]:
import torchvision.models as models

# Load a pretrained ResNet18
# ResNet18 was trained on ImageNet (1000 classes, 1.2M images)
# First run downloads about 45MB - takes a moment

try:
    resnet = models.resnet18(weights='IMAGENET1K_V1')
    print('Loaded pretrained ResNet18')
except Exception:
    resnet = models.resnet18(weights=None)
    print('Created ResNet18 without pretrained weights')

print(f'\nFinal layer: {resnet.fc}')
# resnet.fc outputs 1000 classes (ImageNet) - we will replace this

total = sum(p.numel() for p in resnet.parameters())
print(f'Total parameters: {total:,}')


Loaded pretrained ResNet18

Final layer: Linear(in_features=512, out_features=1000, bias=True)
Total parameters: 11,689,512


In [10]:
# Strategy 1: Feature Extraction
# Freeze all pretrained layers. Train only the new head.

try:
    model_fe = models.resnet18(weights='IMAGENET1K_V1')
except Exception:
    model_fe = models.resnet18(weights=None)

# Freeze everything: requires_grad=False means do not update these during training
for param in model_fe.parameters():
    param.requires_grad = False

# Replace the final layer with one for our task
# resnet.fc.in_features = 512 (coming from the backbone)
num_our_classes = 5
model_fe.fc = nn.Linear(model_fe.fc.in_features, num_our_classes)
# New layers have requires_grad=True by default - only this layer will train

model_fe = model_fe.to(device)

trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_fe.parameters())
print(f'Strategy 1 - Feature Extraction:')
print(f'  Total parameters:     {total:,}')
print(f'  Trainable parameters: {trainable:,}  (only the new head)')
print(f'  Frozen parameters:    {total - trainable:,}')


Strategy 1 - Feature Extraction:
  Total parameters:     11,179,077
  Trainable parameters: 2,565  (only the new head)
  Frozen parameters:    11,176,512


In [11]:
# Strategy 2: Full Fine-Tuning
# Unfreeze all layers and train everything.
# Use a very small learning rate to avoid destroying pretrained features.

try:
    model_ft = models.resnet18(weights='IMAGENET1K_V1')
except Exception:
    model_ft = models.resnet18(weights=None)

model_ft.fc = nn.Linear(model_ft.fc.in_features, num_our_classes)
model_ft = model_ft.to(device)

# Use different learning rates for different parts:
# Backbone (pretrained): very small LR - preserve learned features
# New head: larger LR - learn our task from scratch
backbone_params = [p for name, p in model_ft.named_parameters() if 'fc' not in name]
head_params     = model_ft.fc.parameters()

optimizer_ft = optim.Adam([
    {'params': backbone_params, 'lr': 1e-5},  # very small for backbone
    {'params': head_params,     'lr': 1e-3},  # normal for new head
])

print('Strategy 2 - Full Fine-Tuning:')
for i, group in enumerate(optimizer_ft.param_groups):
    n = sum(p.numel() for p in group['params'])
    print(f'  Group {i}: {n:,} params at lr={group["lr"]}')


Strategy 2 - Full Fine-Tuning:
  Group 0: 11,176,512 params at lr=1e-05
  Group 1: 2,565 params at lr=0.001


## Section 4 - Advanced Training Techniques

These techniques are used in real projects to train faster, use less memory, and prevent overfitting.

Topics:
1. Early Stopping: stop training when validation loss stops improving
2. Mixed Precision Training: use 16-bit floats to train 2-3x faster on GPU
3. Gradient Accumulation: simulate large batch sizes on small GPUs

In [12]:
# Early Stopping
# Problem: training too long causes overfitting.
# The model starts memorizing training data instead of learning general patterns.
# Solution: monitor validation loss. If it does not improve for a set number of epochs,
# stop training and restore the best weights.

class EarlyStopping:
    """
    Stops training when validation loss stops improving.

    patience: how many epochs to wait before stopping
    delta: minimum change to count as improvement
    path: where to save the best model weights
    """

    def __init__(self, patience=7, delta=0.001, path='/tmp/best_model.pth'):
        self.patience    = patience
        self.delta       = delta
        self.path        = path
        self.counter     = 0
        self.best_loss   = None
        self.early_stop  = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self._save(model)

        elif val_loss < self.best_loss - self.delta:
            # Validation loss improved
            print(f'  Val loss improved: {self.best_loss:.4f} -> {val_loss:.4f}')
            self.best_loss = val_loss
            self._save(model)
            self.counter = 0

        else:
            # No improvement
            self.counter += 1
            print(f'  No improvement. Patience: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                print('  Early stopping triggered!')

    def _save(self, model):
        torch.save(model.state_dict(), self.path)

    def load_best(self, model):
        model.load_state_dict(torch.load(self.path, map_location='cpu'))
        return model


print('EarlyStopping class defined.')
print('''
Usage in a training loop:

    early_stopper = EarlyStopping(patience=5)

    for epoch in range(max_epochs):
        train_one_epoch(...)
        val_loss = evaluate(...)
        early_stopper(val_loss, model)
        if early_stopper.early_stop:
            break

    model = early_stopper.load_best(model)
''')


EarlyStopping class defined.

Usage in a training loop:

    early_stopper = EarlyStopping(patience=5)

    for epoch in range(max_epochs):
        train_one_epoch(...)
        val_loss = evaluate(...)
        early_stopper(val_loss, model)
        if early_stopper.early_stop:
            break

    model = early_stopper.load_best(model)



In [13]:
# Mixed Precision Training
from torch.amp import autocast, GradScaler

amp_model     = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
amp_optimizer = optim.Adam(amp_model.parameters(), lr=0.001)
criterion     = nn.CrossEntropyLoss()

# Determine device string
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'

# GradScaler — only beneficial on CUDA; on CPU it's a no-op
scaler = GradScaler(device_str)

fake_x = torch.randn(32, 64).to(device)
fake_y = torch.randint(0, 10, (32,)).to(device)

amp_optimizer.zero_grad()

# autocast — use device string directly
with autocast(device_str):
    output = amp_model(fake_x)
    loss   = criterion(output, fake_y)

scaler.scale(loss).backward()
scaler.step(amp_optimizer)
scaler.update()

print('Mixed precision training step done!')
print(f'Loss: {loss.item():.4f}')
print('Note: benefits are most visible on NVIDIA GPUs (RTX, V100, A100, etc.)')

Mixed precision training step done!
Loss: 2.3287
Note: benefits are most visible on NVIDIA GPUs (RTX, V100, A100, etc.)


In [14]:
# Gradient Accumulation: simulate large batches on small GPUs
# Problem: large batch sizes improve training stability but your GPU may only fit
# 16 or 32 samples at a time.

# Solution: accumulate gradients over N batches, then update once.
# This has the same effect as using a batch N times larger.

# Example: physical batch = 16, accumulate 4 batches -> effective batch = 64

accum_model     = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
accum_optimizer = optim.Adam(accum_model.parameters(), lr=0.001)
criterion       = nn.MSELoss()

accumulation_steps  = 4
physical_batch_size = 16

# Zero gradients at the start, not inside the loop
accum_optimizer.zero_grad()

for step in range(accumulation_steps):
    x = torch.randn(physical_batch_size, 10).to(device)
    y = torch.randn(physical_batch_size, 1).to(device)

    pred = accum_model(x)
    loss = criterion(pred, y)

    # Divide loss by accumulation_steps to average the gradients
    loss = loss / accumulation_steps

    # .backward() adds to existing gradients (does not zero them)
    loss.backward()

    print(f'Step {step+1}/{accumulation_steps}: loss = {(loss * accumulation_steps).item():.4f}')

# Now update weights using all accumulated gradients
accum_optimizer.step()
accum_optimizer.zero_grad()

print(f'\nWeight update done!')
print(f'Effective batch size: {physical_batch_size} x {accumulation_steps} = {physical_batch_size * accumulation_steps}')


Step 1/4: loss = 0.6387
Step 2/4: loss = 1.1617
Step 3/4: loss = 0.8212
Step 4/4: loss = 0.4525

Weight update done!
Effective batch size: 16 x 4 = 64


In [15]:
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    learning_rate:    float = 0.001
    num_epochs:       int   = 50
    patience:         int   = 10        # for early stopping
    min_delta:        float = 0.001     # minimum improvement to count
    use_scheduler:    bool  = True
    scheduler_type:   str   = 'cosine'  # 'step' or 'cosine'
    checkpoint_path:  str   = '/tmp/best_model.pth'


class Trainer:
    """
    A reusable training class that handles the full training loop.
    Includes training, validation, early stopping, and scheduler.
    """

    def __init__(self, model, train_loader, val_loader, config):
        self.model       = model.to(device)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.config       = config

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)

        if config.use_scheduler:
            if config.scheduler_type == 'cosine':
                self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    self.optimizer, T_max=config.num_epochs, eta_min=1e-6)
            else:
                self.scheduler = optim.lr_scheduler.StepLR(
                    self.optimizer, step_size=10, gamma=0.5)
        else:
            self.scheduler = None

        self.best_val_loss   = float('inf')
        self.patience_counter = 0
        self.history = {'train_loss': [], 'val_loss': [],
                        'train_acc': [],  'val_acc': [], 'lr': []}

    def _train_epoch(self):
        self.model.train()
        total_loss = 0
        correct    = 0
        total      = 0

        for x, y in self.train_loader:
            x, y = x.to(device), y.to(device)
            self.optimizer.zero_grad()
            out  = self.model(x)
            loss = self.criterion(out, y)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)

        return total_loss / len(self.train_loader), 100 * correct / total

    def _val_epoch(self):
        self.model.eval()
        total_loss = 0
        correct    = 0
        total      = 0

        with torch.no_grad():
            for x, y in self.val_loader:
                x, y = x.to(device), y.to(device)
                out  = self.model(x)
                loss = self.criterion(out, y)

                total_loss += loss.item()
                correct    += (out.argmax(1) == y).sum().item()
                total      += y.size(0)

        return total_loss / len(self.val_loader), 100 * correct / total

    def fit(self):
        print(f'Training for up to {self.config.num_epochs} epochs...')
        print('-' * 70)

        for epoch in range(self.config.num_epochs):
            current_lr = self.optimizer.param_groups[0]['lr']

            train_loss, train_acc = self._train_epoch()
            val_loss,   val_acc   = self._val_epoch()

            if self.scheduler:
                self.scheduler.step()

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            self.history['lr'].append(current_lr)

            print(f'Epoch {epoch+1:3d}/{self.config.num_epochs} | '
                  f'Train: loss={train_loss:.4f} acc={train_acc:.1f}% | '
                  f'Val: loss={val_loss:.4f} acc={val_acc:.1f}% | '
                  f'LR={current_lr:.6f}')

            # Check for improvement and save best model
            if val_loss < self.best_val_loss - self.config.min_delta:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.config.checkpoint_path)
                self.patience_counter = 0
                print(f'  Best model saved (val_loss={val_loss:.4f})')
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.config.patience:
                    print(f'  Early stopping at epoch {epoch+1}')
                    break

        # Restore best weights
        self.model.load_state_dict(torch.load(self.config.checkpoint_path, map_location=device))
        print('-' * 70)
        print(f'Training complete! Best val loss: {self.best_val_loss:.4f}')

        return self.history


print('Trainer class defined!')


Trainer class defined!


## Part 2 Complete

You have now covered both Parts 1 and 2. Here is a summary:

Part 1 (Foundations):
- Tensors: multi-dimensional arrays, the building block
- Autograd: automatic gradient calculation via .backward()
- nn.Module: base class for all neural networks
- Loss Functions: MSE, CrossEntropy, BCE
- Optimizers: Adam, AdamW, SGD
- Training Loop: Forward -> Loss -> Zero Grad -> Backward -> Update
- DataLoaders: efficient batched data loading


Part 2 (Intermediate/Advanced):
- CNNs: Conv2d + BatchNorm + ReLU + MaxPool to learn spatial features
- Residual Blocks: skip connections to train very deep networks
- RNNs/LSTMs: hidden state as memory to process sequences
- Transfer Learning: reuse pretrained models with much less data
- Early Stopping: stop before overfitting, save best weights

